In [ ]:
!pwd
!ls /root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257/2016
!echo ""
!head -n3 /root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257/2016/main_metadata.csv

/content
all_word_counts.csv  main_metadata.csv	radiant_exp_adv.csv
chat.csv	     objectives.csv	radiant_gold_adv.csv
cosmetics.csv	     picks_bans.csv	teamfights.csv
draft_timings.csv    players.csv	teams.csv

"","match_id","barracks_status_dire","barracks_status_radiant","cluster","dire_score","duration","engine","first_blood_time","game_mode","human_players","leagueid","lobby_type","match_seq_num","negative_votes","positive_votes","radiant_score","radiant_win","start_date_time","tower_status_dire","tower_status_radiant","version","replay_salt","series_id","series_type","patch","region","throw","loss","comeback","stomp","replay_url","dire_team_id","radiant_team_id"
0,2049344492,"'00111111'","'00111111'",133,0,2153,1,192,2,10,4194,1,1804607126,63.0,107.0,0,True,"2016-01-02 15:12:19","'0000011100100000'","'0000011110111110'","",1224511497.0,37073.0,1.0,16,3.0,1017.0,16389.0,"","","http://replay133.valve.net/570/2049344492_1224511497.dem.bz2","",""
1,2049487960,"'00111111'","'00111111'"

In [1]:
import kagglehub
import pandas as pd
import os
import zipfile

# 1. Загрузка датасета
print("Идет загрузка датасета (около 100-200 МБ)...")
path = kagglehub.dataset_download("bwandowando/dota-2-pro-league-matches-2023")
print("Путь к файлам:", path)

# 2. Распаковка (если скачался архив)
data_path = path
if path.endswith('.zip'):
    print("Обнаружен архив, распаковываем...")
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall('/content/kaggle_dataset')
    data_path = '/content/kaggle_dataset'

print("Содержимое папки:", os.listdir(data_path))


Идет загрузка датасета (около 100-200 МБ)...
Путь к файлам: /root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257
Содержимое папки: ['202605', '2018', '202606', '2021', '2022', '202608', '2019', '202603', '202602', '2016', 'Images', '2017', '2024', '2023', '2020', '202604', '2025', '202607', '202609', 'Constants', '202601']


In [ ]:
path = "/root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257"

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

# --- 0. ПУТИ ---
base = path
year_dir = os.path.join(base, '2023')

# --- 1. ЗАГРУЗКА ---
print("Загрузка main_metadata.csv...")
df_match = pd.read_csv(os.path.join(year_dir, 'main_metadata.csv'))

print("Загрузка players.csv...")
df_players = pd.read_csv(os.path.join(year_dir, 'players.csv'))

print("match:", df_match.shape, "| players:", df_players.shape)

# --- 2. СБОРКА ДРАФТА ---
# player_slot: 0-4 Radiant, 128-132 Dire
# Группируем по match_id и собираем список героев для каждой команды

radiant_heroes = (
    df_players[df_players['player_slot'] < 5]
    .sort_values(['match_id', 'player_slot'])
    .groupby('match_id')['hero_id']
    .apply(list)
    .to_dict()
)

dire_heroes = (
    df_players[df_players['player_slot'] >= 128]
    .sort_values(['match_id', 'player_slot'])
    .groupby('match_id')['hero_id']
    .apply(list)
    .to_dict()
)

# Добавляем колонки r1..r5 и d1..d5
# Берём только матчи, где есть ровно 5 героев с каждой стороны
valid_matches = set(radiant_heroes) & set(dire_heroes)
valid_matches = {m for m in valid_matches if len(radiant_heroes[m]) == 5 and len(dire_heroes[m]) == 5}

df_match = df_match[df_match['match_id'].isin(valid_matches)].copy()

for i in range(5):
    df_match[f'r{i+1}_hero'] = df_match['match_id'].map(lambda mid: radiant_heroes[mid][i])
    df_match[f'd{i+1}_hero'] = df_match['match_id'].map(lambda mid: dire_heroes[mid][i])

# --- 3. ПОДГОТОВКА ДАННЫХ ---
X_class = df_match[[f'r{i}_hero' for i in range(1, 6)] + [f'd{i}_hero' for i in range(1, 6)]]
y_class = df_match['radiant_win']

# Регрессия — из players.csv
# В players.csv уже есть колонка 'kda', но пересчитаем для прозрачности
df_players_reg = df_players[df_players['leaver_status'] == 0].copy()
df_players_reg['kda_calc'] = (df_players_reg['kills'] + df_players_reg['assists']) / (df_players_reg['deaths'] + 1)
regression_features = ['gold_per_min', 'xp_per_min', 'hero_damage', 'tower_damage', 'level']
X_reg = df_players_reg[regression_features]
y_reg = df_players_reg['kda_calc']

print("Классификация: X", X_class.shape, "| y", y_class.shape)
print("Регрессия:     X", X_reg.shape, "| y", y_reg.shape)

Загрузка main_metadata.csv...
Загрузка players.csv...


/tmp/ipykernel_11715/4191250381.py:18: DtypeWarning: Columns (67,70,71,124,143) have mixed types. Specify dtype option on import or set low_memory=False.
  df_players = pd.read_csv(os.path.join(year_dir, 'players.csv'))


match: (29810, 43) | players: (298031, 149)
Классификация: X (29802, 10) | y (29802,)
Регрессия:     X (271814, 5) | y (271814,)


In [ ]:
# --- 4. БЕЙЗЛАЙН ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)
X_train_r_scaled = scaler.fit_transform(X_train_r)
X_test_r_scaled = scaler.transform(X_test_r)

knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train_c_scaled, y_train_c)
y_pred_c = knn_clf.predict(X_test_c_scaled)
y_pred_c_proba = knn_clf.predict_proba(X_test_c_scaled)[:, 1]

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train_r_scaled, y_train_r)
y_pred_r = knn_reg.predict(X_test_r_scaled)

# --- 5. ОЦЕНКА ---
print("\n=== РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ (KNN Baseline) ===")
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test_c, y_pred_c_proba):.3f}")
print(f"F1-score: {f1_score(y_test_c, y_pred_c):.3f}")

print("\n=== РЕЗУЛЬТАТЫ РЕГРЕССИИ (KNN Baseline) ===")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.3f}")
print(f"MAE: {mean_absolute_error(y_test_r, y_pred_r):.3f}")


=== РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ (KNN Baseline) ===
Accuracy: 0.499
ROC-AUC: 0.496
F1-score: 0.504

=== РЕЗУЛЬТАТЫ РЕГРЕССИИ (KNN Baseline) ===
RMSE: 3.895
MAE: 2.304


In [ ]:
# ============================================================
# ПУНКТ 3. УЛУЧШЕНИЕ БЕЙЗЛАЙНА — ПРОВЕРКА ГИПОТЕЗ
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline

# Фиксируем бейзлайн для сравнения
baseline = {
    'clf': {'Accuracy': 0.499, 'ROC-AUC': 0.496, 'F1': 0.504},
    'reg': {'RMSE': 3.895, 'MAE': 2.304},
}

# ---------- ГИПОТЕЗА 1: One-hot encoding героев (классификация) ----------
print("=" * 60)
print("ГИПОТЕЗА 1: One-hot encoding героев")
print("=" * 60)

# One-hot по всем 10 позициям драфта
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe = ohe.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_ohe, y_class, test_size=0.2, random_state=42, stratify=y_class
)

knn_clf_ohe = KNeighborsClassifier(n_neighbors=5)
knn_clf_ohe.fit(X_train_c, y_train_c)
y_pred = knn_clf_ohe.predict(X_test_c)
y_proba = knn_clf_ohe.predict_proba(X_test_c)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred):.3f}")
print(f"(бейзлайн: Acc 0.499, AUC 0.496, F1 0.504)")

# ---------- ГИПОТЕЗА 2: Косинусная метрика (классификация) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 2: Косинусная метрика (на one-hot признаках)")
print("=" * 60)

knn_clf_cos = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn_clf_cos.fit(X_train_c, y_train_c)
y_pred = knn_clf_cos.predict(X_test_c)
y_proba = knn_clf_cos.predict_proba(X_test_c)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred):.3f}")

# ---------- ГИПОТЕЗА 3: Подбор k + weights='distance' (классификация) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 3: Подбор k и weights на кросс-валидации")
print("=" * 60)

param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
}
grid = GridSearchCV(
    KNeighborsClassifier(metric='cosine'),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
grid.fit(X_train_c, y_train_c)

print("Лучшие параметры:", grid.best_params_)
print(f"Лучший CV ROC-AUC: {grid.best_score_:.3f}")

best_clf = grid.best_estimator_
y_pred = best_clf.predict(X_test_c)
y_proba = best_clf.predict_proba(X_test_c)[:, 1]
print(f"Test Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"Test ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"Test F1:       {f1_score(y_test_c, y_pred):.3f}")

# ---------- ГИПОТЕЗА 4: Удаление выбросов KDA (регрессия) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 4: Удаление выбросов KDA (топ-1%)")
print("=" * 60)

threshold = y_reg.quantile(0.99)
print(f"Порог 99-го перцентиля KDA: {threshold:.2f}")
print(f"Выбросов: {(y_reg > threshold).sum()} из {len(y_reg)}")

mask = y_reg <= threshold
X_reg_clean = X_reg[mask]
y_reg_clean = y_reg[mask]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

knn_reg_clean = KNeighborsRegressor(n_neighbors=5)
knn_reg_clean.fit(X_train_r_scaled, y_train_r)
y_pred = knn_reg_clean.predict(X_test_r_scaled)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f}")
print(f"MAE:  {mean_absolute_error(y_test_r, y_pred):.3f}")
print(f"(бейзлайн: RMSE 3.895, MAE 2.304)")

# ---------- ГИПОТЕЗА 5: weights='distance' (регрессия) ----------
print("\n" + "=" * 60)
print("ГИПОТЕЗА 5: weights='distance' (регрессия, на очищенных данных)")
print("=" * 60)

knn_reg_dist = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn_reg_dist.fit(X_train_r_scaled, y_train_r)
y_pred = knn_reg_dist.predict(X_test_r_scaled)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f}")
print(f"MAE:  {mean_absolute_error(y_test_r, y_pred):.3f}")

ГИПОТЕЗА 1: One-hot encoding героев
Accuracy: 0.514
ROC-AUC:  0.516
F1:       0.521
(бейзлайн: Acc 0.499, AUC 0.496, F1 0.504)

ГИПОТЕЗА 2: Косинусная метрика (на one-hot признаках)
Accuracy: 0.516
ROC-AUC:  0.516
F1:       0.521

ГИПОТЕЗА 3: Подбор k и weights на кросс-валидации
Лучшие параметры: {'n_neighbors': 11, 'weights': 'distance'}
Лучший CV ROC-AUC: 0.512
Test Accuracy: 0.513
Test ROC-AUC:  0.520
Test F1:       0.526

ГИПОТЕЗА 4: Удаление выбросов KDA (топ-1%)
Порог 99-го перцентиля KDA: 23.00
Выбросов: 2686 из 271814
RMSE: 3.160
MAE:  2.017
(бейзлайн: RMSE 3.895, MAE 2.304)

ГИПОТЕЗА 5: weights='distance' (регрессия, на очищенных данных)
RMSE: 3.174
MAE:  2.025


In [ ]:
# ============================================================
# ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН
# ============================================================

print("=" * 60)
print("ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — КЛАССИФИКАЦИЯ")
print("=" * 60)

ohe_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe_final = ohe_final.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_ohe_final, y_class, test_size=0.2, random_state=42, stratify=y_class
)

final_clf = KNeighborsClassifier(n_neighbors=11, weights='distance', metric='cosine')
final_clf.fit(X_train_c, y_train_c)
y_pred = final_clf.predict(X_test_c)
y_proba = final_clf.predict_proba(X_test_c)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_c, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_c, y_proba):.3f}")
print(f"F1:       {f1_score(y_test_c, y_pred):.3f}")
print(f"(бейзлайн: Acc 0.499, AUC 0.496, F1 0.504)")

print("\n" + "=" * 60)
print("ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — РЕГРЕССИЯ")
print("=" * 60)

threshold = y_reg.quantile(0.99)
mask = y_reg <= threshold
X_reg_clean = X_reg[mask]
y_reg_clean = y_reg[mask]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

final_reg = KNeighborsRegressor(n_neighbors=5)  # без weights='distance'
final_reg.fit(X_train_r_scaled, y_train_r)
y_pred = final_reg.predict(X_test_r_scaled)

print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f}")
print(f"MAE:  {mean_absolute_error(y_test_r, y_pred):.3f}")
print(f"(бейзлайн: RMSE 3.895, MAE 2.304)")

ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — КЛАССИФИКАЦИЯ
Accuracy: 0.513
ROC-AUC:  0.520
F1:       0.526
(бейзлайн: Acc 0.499, AUC 0.496, F1 0.504)

ИТОГОВЫЙ УЛУЧШЕННЫЙ БЕЙЗЛАЙН — РЕГРЕССИЯ
RMSE: 3.160
MAE:  2.017
(бейзлайн: RMSE 3.895, MAE 2.304)


## Перезапуск обучения с оптимизацией по памяти чтобы не падать из-за нехватки ОЗУ

In [2]:
import os
import gc
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

# --- 0. ПУТИ --- base = path
base = '/root/.cache/kagglehub/datasets/bwandowando/dota-2-pro-league-matches-2023/versions/257'
year_dir = os.path.join(base, '2023')

# --- 1. ЗАГРУЗКА ТОЛЬКО НУЖНЫХ КОЛОНОК ---
print("Загрузка main_metadata.csv (только нужные колонки)...")
df_match = pd.read_csv(
    os.path.join(year_dir, 'main_metadata.csv'),
    usecols=['match_id', 'radiant_win']
)

print("Загрузка players.csv (только нужные колонки)...")
players_cols = [
    'match_id', 'player_slot', 'hero_id', 'leaver_status',
    'kills', 'deaths', 'assists',
    'gold_per_min', 'xp_per_min', 'hero_damage', 'tower_damage', 'level'
]
df_players = pd.read_csv(
    os.path.join(year_dir, 'players.csv'),
    usecols=players_cols
)

print("match:", df_match.shape, "| players:", df_players.shape)

# --- 2. СБОРКА ДРАФТА ---
radiant_heroes = (
    df_players[df_players['player_slot'] < 5]
    .sort_values(['match_id', 'player_slot'])
    .groupby('match_id')['hero_id']
    .apply(list)
    .to_dict()
)
dire_heroes = (
    df_players[df_players['player_slot'] >= 128]
    .sort_values(['match_id', 'player_slot'])
    .groupby('match_id')['hero_id']
    .apply(list)
    .to_dict()
)

valid_matches = set(radiant_heroes) & set(dire_heroes)
valid_matches = {m for m in valid_matches
                 if len(radiant_heroes[m]) == 5 and len(dire_heroes[m]) == 5}

df_match = df_match[df_match['match_id'].isin(valid_matches)].copy()

for i in range(5):
    df_match[f'r{i+1}_hero'] = df_match['match_id'].map(lambda mid: radiant_heroes[mid][i])
    df_match[f'd{i+1}_hero'] = df_match['match_id'].map(lambda mid: dire_heroes[mid][i])

# Освобождаем словари — они больше не нужны
del radiant_heroes, dire_heroes
gc.collect()

# --- 3. КЛАССИФИКАЦИЯ: X, y ---
X_class = df_match[[f'r{i}_hero' for i in range(1, 6)] + [f'd{i}_hero' for i in range(1, 6)]]
y_class = df_match['radiant_win']

# --- 4. РЕГРЕССИЯ: X, y ---
df_players_reg = df_players[df_players['leaver_status'] == 0].copy()
df_players_reg['kda_calc'] = (df_players_reg['kills'] + df_players_reg['assists']) / (df_players_reg['deaths'] + 1)

regression_features = ['gold_per_min', 'xp_per_min', 'hero_damage', 'tower_damage', 'level']
X_reg = df_players_reg[regression_features]
y_reg = df_players_reg['kda_calc']

# Освобождаем df_players — он больше не нужен (X_reg и y_reg уже срезы)
del df_players, df_players_reg
gc.collect()

print("Классификация: X", X_class.shape, "| y", y_class.shape)
print("Регрессия:     X", X_reg.shape, "| y", y_reg.shape)

# --- 5. ПОДВЫБОРКА ДЛЯ MyKNN (чтобы не упасть по памяти) ---
# MyKNN строит матрицу (n_test, n_train, n_features) — на полном годе это не влезет.
# Для сравнения MyKNN со sklearn достаточно 20% данных.
sample_match_ids = df_match['match_id'].sample(frac=0.2, random_state=42)
mask_match = df_match['match_id'].isin(sample_match_ids)

X_class_small = X_class[mask_match]
y_class_small = y_class[mask_match]

# Для регрессии — подвыборка по строкам X_reg
sample_idx = X_reg.sample(frac=0.2, random_state=42).index
X_reg_small = X_reg.loc[sample_idx]
y_reg_small = y_reg.loc[sample_idx]

print("Подвыборка для MyKNN:")
print("  Классификация: X", X_class_small.shape, "| y", y_class_small.shape)
print("  Регрессия:     X", X_reg_small.shape, "| y", y_reg_small.shape)

# --- 6. БЕЙЗЛАЙН (sklearn) — на полных данных, для фиксации чисел ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)
X_train_r_scaled = scaler.fit_transform(X_train_r)
X_test_r_scaled = scaler.transform(X_test_r)

knn_clf = KNeighborsClassifier(n_neighbors=5).fit(X_train_c_scaled, y_train_c)
y_pred_c = knn_clf.predict(X_test_c_scaled)
y_proba_c = knn_clf.predict_proba(X_test_c_scaled)[:, 1]

knn_reg = KNeighborsRegressor(n_neighbors=5).fit(X_train_r_scaled, y_train_r)
y_pred_r = knn_reg.predict(X_test_r_scaled)

print("\n=== БЕЙЗЛАЙН sklearn (для сверки) ===")
print(f"CLF  Acc: {accuracy_score(y_test_c, y_pred_c):.3f} | "
      f"AUC: {roc_auc_score(y_test_c, y_proba_c):.3f} | "
      f"F1: {f1_score(y_test_c, y_pred_c):.3f}")
print(f"REG  RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, y_pred_r):.3f}")

# --- 7. УЛУЧШЕННЫЙ БЕЙЗЛАЙН (sklearn) — для сверки с п. 3 ---
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe = ohe.fit_transform(X_class)

X_train_c2, X_test_c2, y_train_c2, y_test_c2 = train_test_split(
    X_class_ohe, y_class, test_size=0.2, random_state=42, stratify=y_class
)

clf_improved = KNeighborsClassifier(n_neighbors=11, weights='distance', metric='cosine')
clf_improved.fit(X_train_c2, y_train_c2)
y_pred_c2 = clf_improved.predict(X_test_c2)
y_proba_c2 = clf_improved.predict_proba(X_test_c2)[:, 1]

threshold = y_reg.quantile(0.99)
mask_clean = y_reg <= threshold
X_reg_clean = X_reg[mask_clean]
y_reg_clean = y_reg[mask_clean]

X_train_r2, X_test_r2, y_train_r2, y_test_r2 = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)
scaler_r2 = StandardScaler()
X_train_r2_scaled = scaler_r2.fit_transform(X_train_r2)
X_test_r2_scaled = scaler_r2.transform(X_test_r2)

reg_improved = KNeighborsRegressor(n_neighbors=5).fit(X_train_r2_scaled, y_train_r2)
y_pred_r2 = reg_improved.predict(X_test_r2_scaled)

print("\n=== УЛУЧШЕННЫЙ БЕЙЗЛАЙН sklearn (для сверки) ===")
print(f"CLF  Acc: {accuracy_score(y_test_c2, y_pred_c2):.3f} | "
      f"AUC: {roc_auc_score(y_test_c2, y_proba_c2):.3f} | "
      f"F1: {f1_score(y_test_c2, y_pred_c2):.3f}")
print(f"REG  RMSE: {np.sqrt(mean_squared_error(y_test_r2, y_pred_r2)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r2, y_pred_r2):.3f}")

# --- 8. ФИНАЛЬНАЯ УБОРКА ---
del X_class_ohe, X_reg_clean, df_match
gc.collect()
print("\nГотово. Переменные для MyKNN: X_class_small, y_class_small, X_reg_small, y_reg_small")

Загрузка main_metadata.csv (только нужные колонки)...
Загрузка players.csv (только нужные колонки)...
match: (29810, 2) | players: (298031, 12)
Классификация: X (29802, 10) | y (29802,)
Регрессия:     X (271814, 5) | y (271814,)
Подвыборка для MyKNN:
  Классификация: X (5960, 10) | y (5960,)
  Регрессия:     X (54363, 5) | y (54363,)

=== БЕЙЗЛАЙН sklearn (для сверки) ===
CLF  Acc: 0.499 | AUC: 0.496 | F1: 0.504
REG  RMSE: 3.895 | MAE: 2.304

=== УЛУЧШЕННЫЙ БЕЙЗЛАЙН sklearn (для сверки) ===
CLF  Acc: 0.513 | AUC: 0.520 | F1: 0.526
REG  RMSE: 3.160 | MAE: 2.017

Готово. Переменные для MyKNN: X_class_small, y_class_small, X_reg_small, y_reg_small


## 5. Имплементация алгоритма KNN (с нуля)
Класс `MyKNN` на numpy: `fit`, `predict`, `predict_proba`. Поддержка `euclidean`/`cosine`, `uniform`/`distance`, батчевый `predict`.

По заданию (п. 4) реализуем собственный класс `MyKNN` на numpy — без использования `sklearn.neighbors`.

**Что поддерживает `MyKNN`:**
- Классификация (мажоритарное голосование) и регрессия (среднее по соседям).
- Метрики расстояния: `euclidean` и `cosine`.
- Взвешивание соседей: `uniform` (все равны) и `distance` (вес обратно пропорционален расстоянию).
- Параметр `n_neighbors`.

**Как работает:**
- `fit` — просто запоминает train-выборку (KNN — ленивый алгоритм, «модели» как таковой нет).
- `predict` — для каждого тестового объекта считает расстояния до всех train-объектов, берёт `k` ближайших и агрегирует их ответы (голосование для классификации, среднее для регрессии).

**Зачем:** сравнить свою реализацию с sklearn — убедиться, что результаты совпадают, и понять, за счёт чего sklearn быстрее (оптимизации на Cython, KD-деревья).

In [4]:
import numpy as np

class MyKNN:
    """
    Собственная реализация KNN на numpy.
    task: 'classification' или 'regression'
    metric: 'euclidean' или 'cosine'
    weights: 'uniform' или 'distance'
    batch_size: размер батча при predict (защита от OOM)
    """
    def __init__(self, n_neighbors=5, task='classification',
                 metric='euclidean', weights='uniform', batch_size=500):
        self.n_neighbors = n_neighbors
        self.task = task
        self.metric = metric
        self.weights = weights
        self.batch_size = batch_size

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=np.float64)
        self.y_train = np.asarray(y)
        return self

    def _distances(self, X):
        """Матрица расстояний (n_test, n_train) без 3D-массива."""
        X = np.asarray(X, dtype=np.float64)
        if self.metric == 'euclidean':
            # ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a·b
            X_sq = np.sum(X ** 2, axis=1, keepdims=True)                 # (n_test, 1)
            T_sq = np.sum(self.X_train ** 2, axis=1)[np.newaxis, :]      # (1, n_train)
            cross = X @ self.X_train.T                                   # (n_test, n_train)
            dist_sq = X_sq + T_sq - 2 * cross
            return np.sqrt(np.maximum(dist_sq, 0))
        elif self.metric == 'cosine':
            X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
            T_norm = self.X_train / (np.linalg.norm(self.X_train, axis=1, keepdims=True) + 1e-12)
            return 1 - X_norm @ T_norm.T
        else:
            raise ValueError(f"Unknown metric: {self.metric}")

    def _neighbor_weights(self, dists):
        if self.weights == 'uniform':
            return np.ones_like(dists)
        elif self.weights == 'distance':
            return 1.0 / (dists + 1e-12)
        else:
            raise ValueError(f"Unknown weights: {self.weights}")

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        n_test = X.shape[0]
        out = []
        for start in range(0, n_test, self.batch_size):
            X_batch = X[start:start + self.batch_size]
            dists = self._distances(X_batch)                       # (batch, n_train)
            idx = np.argsort(dists, axis=1)[:, :self.n_neighbors]
            neighbor_dists = np.take_along_axis(dists, idx, axis=1)
            neighbor_labels = self.y_train[idx]
            w = self._neighbor_weights(neighbor_dists)

            if self.task == 'regression':
                out.append(np.sum(neighbor_labels * w, axis=1) / np.sum(w, axis=1))
            else:
                for i in range(X_batch.shape[0]):
                    votes = {}
                    for label, weight in zip(neighbor_labels[i], w[i]):
                        votes[label] = votes.get(label, 0.0) + weight
                    out.append(max(votes, key=votes.get))

        if self.task == 'regression':
            return np.concatenate(out)
        return np.array(out, dtype=self.y_train.dtype)

    def predict_proba(self, X):
        """Только для классификации."""
        X = np.asarray(X, dtype=np.float64)
        n_test = X.shape[0]
        classes = np.unique(self.y_train)
        proba = np.zeros((n_test, len(classes)))
        for start in range(0, n_test, self.batch_size):
            X_batch = X[start:start + self.batch_size]
            dists = self._distances(X_batch)
            idx = np.argsort(dists, axis=1)[:, :self.n_neighbors]
            neighbor_dists = np.take_along_axis(dists, idx, axis=1)
            neighbor_labels = self.y_train[idx]
            w = self._neighbor_weights(neighbor_dists)
            for ci, c in enumerate(classes):
                mask = (neighbor_labels == c)
                proba[start:start + self.batch_size, ci] = \
                    np.sum(w * mask, axis=1) / np.sum(w, axis=1)
        return proba

## 6. Сравнение MyKNN со sklearn на бейзлайне

Сравнение MyKNN со sklearn на тех же данных и параметрах, что в п. 2.

Прогоняем `MyKNN` на тех же данных и с теми же параметрами, что и `sklearn` в бейзлайне (п. 2): `k=5`, `euclidean`, `uniform`, признаки — сырые `hero_id` для классификации и 5 числовых признаков для регрессии, `StandardScaler` для регрессии.

Ожидаем: результаты совпадут с sklearn (различия — в 4-м знаке после запятой из-за порядка сортировки при равных расстояниях).

In [5]:
print("=" * 60)
print("MyKNN НА БЕЙЗЛАЙНЕ (п. 4a–4e)")
print("=" * 60)

# --- Классификация ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

my_clf = MyKNN(n_neighbors=5, task='classification', metric='euclidean', weights='uniform')
my_clf.fit(X_train_c.values, y_train_c.values)
y_pred = my_clf.predict(X_test_c.values)
y_proba = my_clf.predict_proba(X_test_c.values)[:, 1]

print("\n--- Классификация: MyKNN vs sklearn ---")
print(f"MyKNN   Accuracy: {accuracy_score(y_test_c, y_pred):.3f} | "
      f"ROC-AUC: {roc_auc_score(y_test_c, y_proba):.3f} | "
      f"F1: {f1_score(y_test_c, y_pred):.3f}")
print(f"sklearn Accuracy: 0.499 | ROC-AUC: 0.496 | F1: 0.504")

# --- Регрессия ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

my_reg = MyKNN(n_neighbors=5, task='regression', metric='euclidean', weights='uniform')
my_reg.fit(X_train_r_scaled, y_train_r.values)
y_pred = my_reg.predict(X_test_r_scaled)

print("\n--- Регрессия: MyKNN vs sklearn ---")
print(f"MyKNN   RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, y_pred):.3f}")
print(f"sklearn RMSE: 3.895 | MAE: 2.304")

MyKNN НА БЕЙЗЛАЙНЕ (п. 4a–4e)

--- Классификация: MyKNN vs sklearn ---
MyKNN   Accuracy: 0.500 | ROC-AUC: 0.495 | F1: 0.505
sklearn Accuracy: 0.499 | ROC-AUC: 0.496 | F1: 0.504

--- Регрессия: MyKNN vs sklearn ---
MyKNN   RMSE: 3.895 | MAE: 2.304
sklearn RMSE: 3.895 | MAE: 2.304


## 7. MyKNN с техниками улучшенного бейзлайна (п. 4f–4j)

Сравнение MyKNN со sklearn на улучшенной конфигурации из п. 3.

Добавляем в `MyKNN` техники из п. 3:
- **Классификация:** one-hot encoding героев + `metric='cosine'` + `k=11` + `weights='distance'`.
- **Регрессия:** удаление выбросов KDA (топ-1%) + `k=5` + `weights='uniform'` (в п. 3 `distance` не подтвердилась).

Сравниваем с результатами sklearn из улучшенного бейзлайна (п. 3).

In [6]:
print("=" * 60)
print("MyKNN С ТЕХНИКАМИ УЛУЧШЕННОГО БЕЙЗЛАЙНА (п. 4f–4j)")
print("=" * 60)

# --- Классификация: one-hot + cosine + k=11 + distance ---
ohe_my = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_class_ohe = ohe_my.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_ohe, y_class, test_size=0.2, random_state=42, stratify=y_class
)

my_clf_best = MyKNN(n_neighbors=11, task='classification', metric='cosine', weights='distance')
my_clf_best.fit(X_train_c, y_train_c.values)
y_pred = my_clf_best.predict(X_test_c)
y_proba = my_clf_best.predict_proba(X_test_c)[:, 1]

print("\n--- Классификация: MyKNN vs sklearn (улучшенный) ---")
print(f"MyKNN   Accuracy: {accuracy_score(y_test_c, y_pred):.3f} | "
      f"ROC-AUC: {roc_auc_score(y_test_c, y_proba):.3f} | "
      f"F1: {f1_score(y_test_c, y_pred):.3f}")
print(f"sklearn Accuracy: 0.513 | ROC-AUC: 0.520 | F1: 0.526")

# --- Регрессия: удаление выбросов + k=5 + uniform ---
threshold = y_reg.quantile(0.99)
mask = y_reg <= threshold
X_reg_clean = X_reg[mask]
y_reg_clean = y_reg[mask]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_clean, y_reg_clean, test_size=0.2, random_state=42
)
scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

my_reg_best = MyKNN(n_neighbors=5, task='regression', metric='euclidean', weights='uniform')
my_reg_best.fit(X_train_r_scaled, y_train_r.values)
y_pred = my_reg_best.predict(X_test_r_scaled)

print("\n--- Регрессия: MyKNN vs sklearn (улучшенный) ---")
print(f"MyKNN   RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred)):.3f} | "
      f"MAE: {mean_absolute_error(y_test_r, y_pred):.3f}")
print(f"sklearn RMSE: 3.160 | MAE: 2.017")

MyKNN С ТЕХНИКАМИ УЛУЧШЕННОГО БЕЙЗЛАЙНА (п. 4f–4j)

--- Классификация: MyKNN vs sklearn (улучшенный) ---
MyKNN   Accuracy: 0.513 | ROC-AUC: 0.520 | F1: 0.526
sklearn Accuracy: 0.513 | ROC-AUC: 0.520 | F1: 0.526

--- Регрессия: MyKNN vs sklearn (улучшенный) ---
MyKNN   RMSE: 3.160 | MAE: 2.017
sklearn RMSE: 3.160 | MAE: 2.017


## 8. Вывод по лабораторной №1

- Бейзлайн KNN зафиксирован (п. 2): Accuracy 0.499, ROC-AUC 0.496, F1 0.504; RMSE 3.895, MAE 2.304.
- Улучшенный бейзлайн (п. 3): Accuracy 0.513, ROC-AUC 0.520, F1 0.526; RMSE 3.160, MAE 2.017.
- Собственная имплементация MyKNN (п. 4) воспроизводит sklearn как на бейзлайне, так и на улучшенной конфигурации — расхождения в пределах 3-го знака.
- Абсолютные значения классификации (~0.51) близки к случайному уровню: задача предсказания победителя только по драфту объективно сложна. In-game признаки (`duration`, `radiant_score`) сознательно не использованы — это была бы утечка данных.